# 08 - Ticker conviction: mentions x sentiment combined

Mentions tell you WHERE the crowd is; sentiment tells you WHICH WAY it
leans. Each alone is weak: mentions are direction-blind (GME calls and
puts look identical), and sentiment LEVELS are bullish-biased (retail
skews long, bullish slang outnumbers bearish). Combined, they say more
than either. Everything here is per TICKER; notebook 09 is the same
analysis per THEME. Four combinations, weakest to strongest:

1. **Bull pressure** = bullish posts minus bearish posts per day
   (= `n_posts x net_bullish`). One number carrying volume AND direction:
   100 posts at +0.2 lean = 20 net bulls; 10 posts at +0.9 = 9. Attention
   without direction scores 0; direction without a crowd stays small.
2. **Conviction z** = 7-day rolling bull pressure, z-scored against the
   ticker's OWN window history. +2 = the crowd is unusually large AND
   unusually bullish for THIS name - comparable across big and small
   tickers (the same self-normalising trick as notebook 03).
3. **Divergence flags** - the interesting moments are when the two
   ingredients DISAGREE: attention surging while sentiment DETERIORATES
   is the classic crowded-top / panic pattern; attention surging while
   sentiment improves is a confirmed swarm; sentiment improving quietly
   is an early-watchlist candidate.
4. **Weekly heatmaps + snail trails** - the momentum map (notebooks
   06/07) is a snapshot of the last 5 days; the heatmap unrolls it over
   every week of the window, and the snail trail traces one ticker's
   monthly PATH through the attention-vs-sentiment plane.

Needs `daily_ticker_sentiment.parquet` from notebook 06 (self-contained
otherwise). Saves `daily_ticker_conviction.parquet` - the candidate
composite signal for the trading signals (notebook 10) and the price
backtest (notebook 11).

In [1]:
import os, sys
ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, ROOT)
print('project root:', ROOT)

project root: C:\Users\alexd\Desktop\GIC\RetailFlow1


In [2]:
# ============ PARAMETERS - edit these ============
SENT_PATH       = os.path.join(ROOT, 'data', 'processed', 'daily_ticker_sentiment.parquet')
CONVICTION_OUT  = os.path.join(ROOT, 'data', 'processed', 'daily_ticker_conviction.parquet')
TICKERS         = []    # e.g. ['GME', 'NVDA']; [] = TOP_N by post volume
TOP_N           = 6
ROLL            = 7     # rolling window for bull pressure
HEATMAP_TICKERS = 20    # rows in the weekly heatmaps
TRAIL_TICKERS   = 4     # snail trails to draw (one figure each)
DIVERGENCE_Z    = 1.0   # attention z above this counts as 'surging'
# ==================================================

In [3]:
# Build bull pressure + conviction z per ticker per day.
import numpy as np
import pandas as pd

if not os.path.exists(SENT_PATH):
    raise FileNotFoundError('daily_ticker_sentiment.parquet not found - run notebook 06 first.')
sent = pd.read_parquet(SENT_PATH)
sent['date'] = pd.to_datetime(sent['date'])
WINDOW = f"{sent['date'].min().date()} to {sent['date'].max().date()}"
print('window:', WINDOW, '|', sent['ticker'].nunique(), 'tickers')

# bull pressure: net bullish POST COUNT (volume x direction in one number)
sent['bull_pressure'] = sent['n_posts'] * sent['net_bullish']

all_days = pd.date_range(sent['date'].min(), sent['date'].max(), freq='D')
wide_bp = (sent.pivot_table(index='date', columns='ticker', values='bull_pressure')
           .reindex(all_days).fillna(0))          # no posts = zero pressure
wide_n  = (sent.pivot_table(index='date', columns='ticker', values='n_posts')
           .reindex(all_days).fillna(0))

roll_bp = wide_bp.rolling(ROLL, min_periods=1).sum()
roll_n  = wide_n.rolling(ROLL, min_periods=1).sum()

# conviction z: each ticker vs its OWN window mean/std of rolled pressure
# TRAILING z (live-parity, same rule as notebook 10): each day scored
# against the PRECEDING 84 days only - no future info, and quiet periods
# are no longer dragged negative by the 2021 volume mania
_mu = roll_bp.rolling(84, min_periods=28).mean()
_sd = roll_bp.rolling(84, min_periods=28).std().replace(0, np.nan)
conviction_z = (roll_bp - _mu) / _sd
# attention z: same idea on volume only (used for the divergence flags)
attention_z = (roll_n - roll_n.mean()) / roll_n.std().replace(0, np.nan)

out = (conviction_z.stack().rename('conviction_z').reset_index()
       .rename(columns={'level_0': 'date'}))
out.to_parquet(CONVICTION_OUT, index=False)
print('saved ->', CONVICTION_OUT)

if TICKERS:
    chosen = TICKERS
else:
    chosen = list(wide_n.sum().sort_values(ascending=False).head(TOP_N).index)
print('analysing:', chosen)

window: 2017-01-01 to 2026-07-18 | 7950 tickers


saved -> C:\Users\alexd\Desktop\GIC\RetailFlow1\data\processed\daily_ticker_conviction.parquet


analysing: ['GME', 'TSLA', 'PLTR', 'BBBY', 'AAPL', 'AMC']


In [4]:
# CONVICTION CHART per ticker + divergence flags.
import matplotlib.pyplot as plt

# sentiment CHANGE: 5-day difference of the rolling bullish share
share = (roll_bp / roll_n.replace(0, np.nan))     # rolled net_bullish share
sent_change = share.diff(5)

for ticker in chosen:
    cz = conviction_z[ticker]
    az = attention_z[ticker]
    dv = sent_change[ticker]

    crowded_top = (az > DIVERGENCE_Z) & (dv < -0.10)   # crowd up, mood down
    swarm       = (az > DIVERGENCE_Z) & (dv > +0.10)   # crowd up, mood up

    fig, ax = plt.subplots(figsize=(12.5, 4.5))
    ax.plot(cz.index, cz, linewidth=1.4, color='steelblue', label='conviction z')
    ax.fill_between(cz.index, 0, cz, where=cz >= 0, alpha=0.15, color='green')
    ax.fill_between(cz.index, 0, cz, where=cz < 0, alpha=0.15, color='red')
    ax.scatter(cz.index[swarm], cz[swarm], marker='^', s=60, color='seagreen',
               zorder=5, label='confirmed swarm (crowd+mood rising)')
    ax.scatter(cz.index[crowded_top], cz[crowded_top], marker='v', s=60,
               color='crimson', zorder=5, label='crowded-top risk (crowd up, mood falling)')
    ax.axhline(0, color='black', linewidth=0.6)
    ax.set_title(f'{ticker} - retail conviction ({ROLL}d bull pressure, z vs own history) | window {WINDOW}')
    ax.set_ylabel('conviction z'); ax.set_xlabel('date')
    ax.legend(loc='upper left'); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

    for label, mask in (('SWARM', swarm), ('CROWDED-TOP', crowded_top)):
        days = list(cz.index[mask].strftime('%Y-%m-%d'))
        if days:
            print(f'{ticker} {label}: {", ".join(days[:8])}' + (' ...' if len(days) > 8 else ''))

C:\Users\alexd\AppData\Local\Temp\ipykernel_58280\3824593327.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


GME SWARM: 2021-02-25, 2021-02-26
GME CROWDED-TOP: 2021-01-28, 2021-03-16


C:\Users\alexd\AppData\Local\Temp\ipykernel_58280\3824593327.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


TSLA SWARM: 2018-08-08, 2020-01-10, 2020-01-26, 2020-02-14, 2020-02-15, 2020-02-16, 2020-02-17, 2020-03-22 ...
TSLA CROWDED-TOP: 2020-01-18, 2020-02-05, 2020-02-06, 2020-02-07, 2020-02-08, 2020-02-09, 2020-02-27, 2020-05-05 ...


C:\Users\alexd\AppData\Local\Temp\ipykernel_58280\3824593327.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


...

BBBY SWARM: 2022-08-29, 2022-08-30, 2022-09-12, 2023-01-13, 2023-01-14, 2023-01-15, 2023-01-16
BBBY CROWDED-TOP: 2022-08-09, 2022-08-12, 2022-09-04, 2022-09-05, 2022-09-06, 2022-09-07, 2022-09-08, 2022-09-09 ...


C:\Users\alexd\AppData\Local\Temp\ipykernel_58280\3824593327.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


AAPL SWARM: 2017-02-04, 2019-01-09, 2020-01-30, 2020-01-31, 2020-02-01, 2020-03-22, 2020-04-18, 2020-04-19 ...
AAPL CROWDED-TOP: 2017-02-01, 2018-08-01, 2018-08-02, 2019-01-04, 2020-07-25, 2020-07-26, 2020-07-27, 2020-07-28 ...


AMC SWARM: 2021-02-26
AMC CROWDED-TOP: 2021-01-28


C:\Users\alexd\AppData\Local\Temp\ipykernel_58280\3824593327.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## Weekly heatmaps - the momentum map unrolled over time

The 06/07 momentum maps show ONE 5-day snapshot. These heatmaps show every
week of the window at once: rows = the most-posted tickers, columns =
weeks, colour = the value that week (grey = too few posts to score).

- **Heatmap 1: sentiment** (weekly net bullish share, red -1 .. green +1).
  Read rows for a ticker's mood over time, columns for market-wide shifts.
- **Heatmap 2: conviction z** - where volume AND direction were unusual
  together. A bright green cell = that ticker's crowd was unusually big
  and bullish that week; deep red = unusually big and bearish.

In [5]:
top = list(wide_n.sum().sort_values(ascending=False).head(HEATMAP_TICKERS).index)

wk_bp = wide_bp[top].resample('W').sum()
wk_n  = wide_n[top].resample('W').sum()
wk_share = (wk_bp / wk_n.replace(0, np.nan)).where(wk_n >= 10)   # grey out thin weeks
wk_cz = ((wk_bp - wk_bp.mean()) / wk_bp.std().replace(0, np.nan))

def heatmap(frame, title, vmin, vmax):
    fig, ax = plt.subplots(figsize=(13, 0.42 * len(top) + 1.5))
    data = frame.T  # rows = tickers, columns = weeks
    im = ax.imshow(data, aspect='auto', cmap='RdYlGn', vmin=vmin, vmax=vmax)
    ax.set_yticks(range(len(data.index)), data.index, fontsize=8)
    ticks = range(0, len(data.columns), max(1, len(data.columns) // 12))
    ax.set_xticks(list(ticks), [data.columns[i].strftime('%Y-%m-%d') for i in ticks],
                  rotation=45, ha='right', fontsize=8)
    ax.set_title(f'{title} | window {WINDOW}')
    fig.colorbar(im, ax=ax, shrink=0.8)
    plt.tight_layout(); plt.show()

heatmap(wk_share, 'Weekly net bullish share (grey = under 10 scored posts)', -1, 1)
heatmap(wk_cz, 'Weekly conviction z (volume x direction vs own history)', -3, 3)

C:\Users\alexd\AppData\Local\Temp\ipykernel_58280\1463076657.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


C:\Users\alexd\AppData\Local\Temp\ipykernel_58280\1463076657.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## Snail trails - a ticker's path through the plane

One dot per month: x = average attention z, y = average net bullish share
that month, connected in time order (darker = more recent, the label is
the month). A healthy swarm walks right AND up; the classic blow-off
pattern walks right while sliding DOWN (crowd still growing, mood already
rolling over) - visible here weeks before it shows in price.

In [6]:
for ticker in chosen[:TRAIL_TICKERS]:
    ax_m = attention_z[ticker].resample('ME').mean()
    share_m = share[ticker].resample('ME').mean()
    trail = pd.DataFrame({'az': ax_m, 'share': share_m}).dropna()
    if len(trail) < 3:
        print('skip', ticker, '- fewer than 3 months of data'); continue

    fig, ax = plt.subplots(figsize=(9, 7))
    shades = np.linspace(0.25, 1.0, len(trail))
    ax.plot(trail['az'], trail['share'], color='gray', linewidth=1, alpha=0.6, zorder=1)
    ax.scatter(trail['az'], trail['share'], s=90, c=shades, cmap='Blues',
               edgecolors='black', linewidths=0.6, zorder=2)
    for i, (idx, r) in enumerate(trail.iterrows()):
        ax.annotate(idx.strftime('%b %y'), (r['az'], r['share']),
                    textcoords='offset points', xytext=(7, 5), fontsize=8)
    ax.axhline(0, color='black', linewidth=0.6)
    ax.axvline(0, color='black', linewidth=0.6)
    ax.margins(0.15)
    ax.set_xlabel('monthly avg attention z (crowd size vs own history)')
    ax.set_ylabel('monthly avg net bullish share')
    ax.set_title(f'{ticker} - monthly path through attention x sentiment | window {WINDOW}')
    ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

C:\Users\alexd\AppData\Local\Temp\ipykernel_58280\3476269636.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


C:\Users\alexd\AppData\Local\Temp\ipykernel_58280\3476269636.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


C:\Users\alexd\AppData\Local\Temp\ipykernel_58280\3476269636.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


C:\Users\alexd\AppData\Local\Temp\ipykernel_58280\3476269636.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## Caveats & what to do with this

- Conviction z inherits BOTH parents' weaknesses: sampling noise from the
  scored-post cap and lexicon bias from VADER. The z-scoring removes the
  constant bullish LEVEL bias per ticker, but a market-wide mood swing
  still moves every ticker at once (that is real information, not a bug).
- Divergence flags are descriptive, not signals, until they survive the
  price backtest: notebook 11 should test conviction_z and the flags the
  same way WEEK2.md tests raw mentions (lags, event study, permutation).
- Everything here uses SCORED posts; with sampling on (notebook 06),
  absolute volumes are proportional estimates.